#import package

In [ ]:
import os
import json
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader


#colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!pwd

#config

In [ ]:
config = {
    'LongBench_path': "/content/drive/MyDrive/LongBench",
    'model_name': "Qwen/Qwen2.5-1.5B",
    'model_limit': 4096,
    'max_length': 4096,
    'use_e': False,
    'batch_size': 2,
    'lr': 2e-5,
    'epochs': 2,
    'output_dir': "/content/qwen_longbench_finetune"
}



In [ ]:
class LongBenchDatasetBuilder:
    """
    LongBench Dataset Builder for fine-tuning.
    - Auto-detect short/long tasks by avg answer length
    - Short tasks: context + answer labels
    - Long tasks: answer-only labels
    - Pads to max_length
    """

    def __init__(self, base_dir, model_name, model_limit=32768, use_e=False, max_length=4096):
        self.base_dir = base_dir
        self.use_e = use_e
        self.max_length = max_length
        self.model_limit = model_limit

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        with open(os.path.join(base_dir, "dataset2prompt.json"), "r", encoding="utf-8") as f:
            self.prompts = json.load(f)

    def compute_global_maxlen(self, task_list):
        """Compute max token length across tasks."""
        global_max = 0
        for task in task_list:
            fname = f"{task}_e.jsonl" if self.use_e else f"{task}.jsonl"
            path = os.path.join(self.base_dir, "data", fname)
            data = load_dataset("json", data_files=path, split="train")
            template = self.prompts[task]
            for ex in data:
                prompt = template.format(context=ex["context"], input=ex["input"])
                ans = ex["answers"][0] if ex.get("answers") else ""
                text = prompt + " " + ans
                token_len = len(self.tokenizer.encode(text, add_special_tokens=False))
                global_max = max(global_max, token_len)
        return min(max(global_max, self.max_length), self.model_limit)

    def load_one(self, task, max_length):
        """Load one task and preprocess."""
        fname = f"{task}_e.jsonl" if self.use_e else f"{task}.jsonl"
        path = os.path.join(self.base_dir, "data", fname)
        dataset = load_dataset("json", data_files=path, split="train")
        template = self.prompts[task]

        # Compute mean answer length
        avg_ans_len = 0
        samples = min(len(dataset), 200)
        for ex in dataset.select(range(samples)):
            ans = ex["answers"][0] if ex.get("answers") else ""
            avg_ans_len += len(self.tokenizer.encode(ans, add_special_tokens=False))
        avg_ans_len /= max(1, samples)
        SHORT_ANSWER = avg_ans_len < 100

        def preprocess(ex):
            prompt = template.format(context=ex["context"], input=ex["input"])
            ans = ex["answers"][0] if ex.get("answers") else ""
            p_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
            a_ids = self.tokenizer.encode(ans, add_special_tokens=False)
            input_ids = (p_ids + a_ids)[-max_length:]

            if SHORT_ANSWER:
                context_window = 256
                if len(a_ids) + context_window < len(input_ids):
                    labels = [-100] * (len(input_ids) - len(a_ids) - context_window)
                    labels += input_ids[-(len(a_ids) + context_window):]
                else:
                    labels = input_ids[:]
            else:
                labels = [-100] * (len(input_ids) - len(a_ids)) + a_ids

            attention_mask = [1] * len(input_ids)
            pad_len = max_length - len(input_ids)
            if pad_len > 0:
                pad_id = self.tokenizer.pad_token_id
                input_ids = [pad_id] * pad_len + input_ids
                labels = [-100] * pad_len + labels
                attention_mask = [0] * pad_len + attention_mask

            return {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "task": task
            }

        return dataset.map(preprocess, remove_columns=dataset.column_names)

    def load_all(self, task_list=None):
        """Load and merge multiple tasks."""
        if task_list is None:
            task_list = list(self.prompts.keys())

        global_max = self.compute_global_maxlen(task_list)
        datasets = [self.load_one(task, global_max) for task in task_list]
        merged = concatenate_datasets(datasets)
        return merged, global_max


In [ ]:

# Build dataset
builder = LongBenchDatasetBuilder(
    base_dir=config['LongBench_path'],
    model_name=config['model_name'],
    model_limit=config['model_limit'],
    use_e=config['use_e'],
    max_length=config['max_length']
)

# Load subset tasks
subset_tasks = ["hotpotqa", "gov_report"]
dataset_subset, max_len_subset = builder.load_all(subset_tasks)

# Collate function
def collate_fn(batch):
    return {
        "input_ids": torch.tensor([b["input_ids"] for b in batch], dtype=torch.long),
        "attention_mask": torch.tensor([b["attention_mask"] for b in batch], dtype=torch.long),
        "labels": torch.tensor([b["labels"] for b in batch], dtype=torch.long)
    }

# Build DataLoader
loader_subset = DataLoader(
    dataset_subset,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=collate_fn
)




# === Optional: Load all tasks ===
'''
dataset_all, max_len_all = builder.load_all()
print(f"Loaded {len(dataset_all)} samples | Max length = {max_len_all}")

loader_all = DataLoader(
    dataset_all,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=collate_fn
)
'''
